# Transit Compliance Bot Build Notebook

This notebook builds the offline artifacts for a citation-first RAG assistant over **49 CFR Part 37** and **49 CFR Part 38**.

With only two source documents, the correct low-latency setup is not model fine-tuning. Instead, this notebook:
- downloads ungated Hugging Face models into the project
- extracts and chunks both PDFs
- creates a CUDA-accelerated embedding index
- saves the manifest and chunk metadata that `app.py` will load later

Artifacts created:
- `artifacts/chroma_db/`
- `artifacts/chunks.jsonl`
- `artifacts/manifest.json`
- `artifacts/models/...`


In [1]:
from __future__ import annotations

import json
import logging
import os
import re
import shutil
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import torch
from chromadb.config import Settings
from huggingface_hub import snapshot_download
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ANONYMIZED_TELEMETRY"] = "FALSE"

logging.getLogger("tensorflow").setLevel(logging.ERROR)
logging.getLogger("absl").setLevel(logging.ERROR)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


def resolve_runtime_device() -> tuple[str, str | None]:
    if not torch.cuda.is_available():
        return "cpu", "CUDA is not available in the current PyTorch runtime."

    try:
        capability = torch.cuda.get_device_capability(0)
        capability_tag = f"sm_{capability[0]}{capability[1]}"
        supported_arches = set(getattr(torch.cuda, "get_arch_list", lambda: [])())
        if supported_arches and capability_tag not in supported_arches:
            return (
                "cpu",
                f"Detected GPU capability {capability_tag}, but this PyTorch build only supports: "
                + ", ".join(sorted(supported_arches)),
            )

        torch.zeros(1, device="cuda")
        return "cuda", None
    except Exception as error:
        return "cpu", f"CUDA initialization failed: {error}"

PROJECT_ROOT = Path.cwd()
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"


@dataclass(frozen=True)
class BuildConfig:
    pdf_catalog: dict[str, str]
    collection_name: str = "fta_transit_regulations"
    embedding_repo_id: str = "BAAI/bge-base-en-v1.5"
    reranker_repo_id: str = "BAAI/bge-reranker-base"
    generator_repo_id: str = "Qwen/Qwen2.5-1.5B-Instruct"
    chunk_size: int = 900
    chunk_overlap: int = 180
    embedding_batch_size: int = 32
    dense_top_k: int = 8
    dense_fetch_k: int = 24
    bm25_top_k: int = 8
    rerank_top_n: int = 5
    max_context_chunks: int = 5


config = BuildConfig(
    pdf_catalog={
        "49 CFR Part 37": "49 CFR Part 37 (up to date as of 3-09-2026).pdf",
        "49 CFR Part 38": "CFR-2024-title49-vol1-part38.pdf",
    }
)

MODEL_ROOT = ARTIFACTS_DIR / "models"
EMBEDDING_DIR = MODEL_ROOT / "embeddings" / config.embedding_repo_id.split("/")[-1]
RERANKER_DIR = MODEL_ROOT / "reranker" / config.reranker_repo_id.split("/")[-1]
GENERATOR_DIR = MODEL_ROOT / "generator" / config.generator_repo_id.split("/")[-1]
VECTOR_DB_DIR = ARTIFACTS_DIR / "chroma_db"
CHUNKS_PATH = ARTIFACTS_DIR / "chunks.jsonl"
MANIFEST_PATH = ARTIFACTS_DIR / "manifest.json"

DEVICE, DEVICE_WARNING = resolve_runtime_device()
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif DEVICE_WARNING:
    print(f"CUDA disabled: {DEVICE_WARNING}")

EMBEDDING_BATCH_SIZE = config.embedding_batch_size if DEVICE == "cuda" else 8


c:\Users\moham\.conda\envs\ml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
CUDA disabled: Detected GPU capability sm_120, but this PyTorch build only supports: sm_50, sm_60, sm_61, sm_70, sm_75, sm_80, sm_86, sm_90


c:\Users\moham\.conda\envs\ml\lib\site-packages\torch\cuda\__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5080 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5080 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [2]:
def normalize_text(text: str) -> str:
    text = text.replace("\u0000", " ")
    text = text.replace("\r", "\n")
    text = re.sub(r"(?<=\w)-\s*\n(?=\w)", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def slugify(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


def ensure_workspace() -> None:
    ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_ROOT.mkdir(parents=True, exist_ok=True)


MODEL_DOWNLOAD_PATTERNS = {
    config.embedding_repo_id: [
        "1_Pooling/config.json",
        "config.json",
        "config_sentence_transformers.json",
        "model.safetensors",
        "modules.json",
        "sentence_bert_config.json",
        "special_tokens_map.json",
        "tokenizer.json",
        "tokenizer_config.json",
        "vocab.txt",
    ],
    config.reranker_repo_id: [
        "config.json",
        "model.safetensors",
        "sentencepiece.bpe.model",
        "special_tokens_map.json",
        "tokenizer.json",
        "tokenizer_config.json",
    ],
    config.generator_repo_id: [
        "config.json",
        "generation_config.json",
        "merges.txt",
        "model.safetensors",
        "tokenizer.json",
        "tokenizer_config.json",
        "vocab.json",
    ],
}


def download_model(repo_id: str, target_dir: Path) -> Path:
    expected_files = MODEL_DOWNLOAD_PATTERNS.get(repo_id, [])
    if target_dir.exists() and all((target_dir / relative_path).exists() for relative_path in expected_files):
        print(f"Using cached model at {target_dir}")
        return target_dir

    target_dir.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {repo_id} -> {target_dir}")
    snapshot_download(
        repo_id=repo_id,
        local_dir=str(target_dir),
        allow_patterns=expected_files or None,
    )
    return target_dir


def load_pdf_pages(source_title: str, pdf_name: str) -> list[Document]:
    pdf_path = PROJECT_ROOT / pdf_name
    if not pdf_path.exists():
        raise FileNotFoundError(f"Missing source PDF: {pdf_path}")

    loader = PyMuPDFLoader(str(pdf_path))
    raw_pages = loader.load()
    source_key = slugify(source_title)
    pages: list[Document] = []

    for page in raw_pages:
        content = normalize_text(page.page_content)
        if not content:
            continue

        page_number = int(page.metadata.get("page", 0)) + 1
        pages.append(
            Document(
                page_content=content,
                metadata={
                    "source_key": source_key,
                    "source_title": source_title,
                    "source_file": pdf_path.name,
                    "source_path": str(pdf_path),
                    "page": page_number,
                    "citation": f"{source_title}, p. {page_number}",
                },
            )
        )

    return pages


def build_chunks(pages: list[Document]) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.chunk_size,
        chunk_overlap=config.chunk_overlap,
        separators=["\n\n", "\n", ". ", "; ", " ", ""],
    )
    split_docs = splitter.split_documents(pages)
    chunks: list[Document] = []

    for index, chunk in enumerate(split_docs, start=1):
        metadata = dict(chunk.metadata)
        first_line = next((line.strip() for line in chunk.page_content.splitlines() if line.strip()), "")
        metadata["chunk_id"] = f"{metadata['source_key']}::p{metadata['page']:03d}::c{index:05d}"
        metadata["section_hint"] = first_line[:160]
        metadata["char_count"] = len(chunk.page_content)
        chunks.append(Document(page_content=chunk.page_content, metadata=metadata))

    return chunks


def save_chunks(chunks: list[Document], output_path: Path) -> None:
    with output_path.open("w", encoding="utf-8") as handle:
        for chunk in chunks:
            record = {
                "page_content": chunk.page_content,
                "metadata": chunk.metadata,
            }
            handle.write(json.dumps(record, ensure_ascii=True) + "\n")


def build_vector_store(chunks: list[Document]) -> Chroma:
    if VECTOR_DB_DIR.exists():
        shutil.rmtree(VECTOR_DB_DIR)

    embeddings = HuggingFaceEmbeddings(
        model_name=str(EMBEDDING_DIR),
        model_kwargs={"device": DEVICE},
        encode_kwargs={
            "normalize_embeddings": True,
            "batch_size": EMBEDDING_BATCH_SIZE,
        },
    )

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        ids=[chunk.metadata["chunk_id"] for chunk in chunks],
        collection_name=config.collection_name,
        persist_directory=str(VECTOR_DB_DIR),
        client_settings=Settings(anonymized_telemetry=False),
    )

    if hasattr(vector_store, "persist"):
        vector_store.persist()

    return vector_store


def write_manifest(all_pages: list[Document], chunks: list[Document]) -> dict:
    document_stats = []
    for title, pdf_name in config.pdf_catalog.items():
        document_stats.append(
            {
                "title": title,
                "file_name": pdf_name,
                "path": str(PROJECT_ROOT / pdf_name),
                "page_count": sum(1 for doc in all_pages if doc.metadata["source_title"] == title),
            }
        )

    manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "paths": {
            "project_root": str(PROJECT_ROOT),
            "artifacts_dir": str(ARTIFACTS_DIR),
            "chunks_path": str(CHUNKS_PATH),
            "vector_db_dir": str(VECTOR_DB_DIR),
        },
        "documents": document_stats,
        "models": {
            "embedding_repo_id": config.embedding_repo_id,
            "embedding_dir": str(EMBEDDING_DIR),
            "reranker_repo_id": config.reranker_repo_id,
            "reranker_dir": str(RERANKER_DIR),
            "generator_repo_id": config.generator_repo_id,
            "generator_dir": str(GENERATOR_DIR),
        },
        "retrieval": {
            "collection_name": config.collection_name,
            "chunk_size": config.chunk_size,
            "chunk_overlap": config.chunk_overlap,
            "dense_top_k": config.dense_top_k,
            "dense_fetch_k": config.dense_fetch_k,
            "bm25_top_k": config.bm25_top_k,
            "rerank_top_n": config.rerank_top_n,
            "max_context_chunks": config.max_context_chunks,
        },
        "generation": {
            "max_new_tokens": 400,
            "temperature": 0.0,
            "do_sample": False,
        },
        "stats": {
            "page_count": len(all_pages),
            "chunk_count": len(chunks),
            "avg_chunk_chars": round(
                sum(len(doc.page_content) for doc in chunks) / max(len(chunks), 1),
                2,
            ),
        },
    }

    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest


In [3]:
ensure_workspace()

download_model(config.embedding_repo_id, EMBEDDING_DIR)
download_model(config.reranker_repo_id, RERANKER_DIR)
download_model(config.generator_repo_id, GENERATOR_DIR)

all_pages: list[Document] = []
for source_title, pdf_name in config.pdf_catalog.items():
    pages = load_pdf_pages(source_title, pdf_name)
    print(f"{source_title}: extracted {len(pages)} pages")
    all_pages.extend(pages)

chunks = build_chunks(all_pages)
print(f"Built {len(chunks)} chunks")

save_chunks(chunks, CHUNKS_PATH)
build_vector_store(chunks)
manifest = write_manifest(all_pages, chunks)

print(f"Artifacts saved under: {ARTIFACTS_DIR}")
print(json.dumps(manifest["stats"], indent=2))

if DEVICE == "cuda":
    torch.cuda.empty_cache()


Using cached model at c:\Users\moham\Desktop\fta chatbot\artifacts\models\embeddings\bge-base-en-v1.5
Using cached model at c:\Users\moham\Desktop\fta chatbot\artifacts\models\reranker\bge-reranker-base
Using cached model at c:\Users\moham\Desktop\fta chatbot\artifacts\models\generator\Qwen2.5-1.5B-Instruct
49 CFR Part 37: extracted 140 pages
49 CFR Part 38: extracted 39 pages
Built 932 chunks



Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Artifacts saved under: c:\Users\moham\Desktop\fta chatbot\artifacts
{
  "page_count": 179,
  "chunk_count": 932,
  "avg_chunk_chars": 795.64
}


In [4]:
# Quick validation: run a retrieval query against the persisted vector store.
embeddings = HuggingFaceEmbeddings(
    model_name=str(EMBEDDING_DIR),
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)

vector_store = Chroma(
    collection_name=config.collection_name,
    persist_directory=str(VECTOR_DB_DIR),
    embedding_function=embeddings,
    client_settings=Settings(anonymized_telemetry=False),
)

sample_query = "What accessibility requirements apply to lifts and ramps on transit vehicles?"
results = vector_store.similarity_search(sample_query, k=3)

for rank, doc in enumerate(results, start=1):
    print(f"{rank}. {doc.metadata['citation']}")
    print(doc.page_content[:500])
    print("-" * 80)


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


1. 49 CFR Part 38, p. 27
protect the eyes of entering and exiting 
passengers. 
[56 FR 45756, Sept. 6, 1991, as amended at 63 
FR 51698, 51702, Sept. 28, 1998] 
§ 38.159
Mobility aid accessibility. 
(a)(1) General. All vehicles covered by 
this subpart shall provide a levelchange mechanism or boarding device 
(e.g., lift or ramp) complying with 
paragraph (b) or (c) of this section and 
sufficient clearances to permit a wheelchair or other mobility aid user to 
reach a securement location. At least 
two securement locat
--------------------------------------------------------------------------------
2. 49 CFR Part 38, p. 11
(a)(1) General. All new light rail vehicles, other than level entry vehicles, 
covered by this subpart shall provide a 
level-change mechanism or boarding 
device (e.g., lift, ramp or bridge plate) 
complying with either paragraph (b) or 
(c) of this section and sufficient clearances to permit at least two wheelchair 
or mobility aid users to reach areas, 
each with